In [35]:
import json
import urllib.parse
import shutil
from pathlib import Path
import numpy as np 
import cv2
import pandas as pd
from matplotlib import pyplot as plt

In [32]:
root = Path("/Users/n-zagainov/kolobok/ml/data/det_spikes2")
images = root / "images"
annots = root / "annot"
save_dir = root / "vis"

In [34]:
color_map = {
    "normal": (0, 255, 0),
    "broken": (0, 0, 255),
    "absent": (255, 0, 0),
    "floating": (0, 255, 255),
    "renewed": (255, 255, 0),
}

for image_path in list(images.iterdir()):
    image_name = image_path.stem
    annot_path = annots / f"{image_name}.json"

    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR_RGB)
    orig_h, orig_w = image.shape[:2]
    with open(annot_path, "r") as f:
        data = json.load(f)

    for annot in data:
        x, y, w, h = annot["x"], annot["y"], annot["w"], annot["h"]
        x_min = int((x - w / 2) * orig_w)
        y_min = int((y - h / 2) * orig_h)
        x_max = int((x + w / 2) * orig_w)
        y_max = int((y + h / 2) * orig_h)

        cv2.rectangle(image, (x_min, y_min), (x_max, y_max), color_map[annot["label"]], 2)

    save_path = save_dir / image_path.name
    save_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(save_path), image)

In [37]:
corrupt_save_dir = root / "corrupt_images"
corrupt_vis_dir = root / "corrupt_vis"

for image_path in list(images.iterdir()):
    vis_path = save_dir / image_path.name
    if not vis_path.exists():
        save_path = corrupt_save_dir / image_path.name
        save_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(image_path, save_path)
        
for vis_path in list(save_dir.iterdir()):
    if "corrupt" in vis_path.name:
        save_path = corrupt_vis_dir / vis_path.name
        save_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(vis_path, save_path)
